# `tensor.tex` — the formula is the program, from Python

*Parse LaTeX-subset expressions at runtime, bind concrete tensors to the AST, and get back a `DynamicTensor` whose values are exactly what the formula says they should be. The Python-side equivalent of C++'s `R"(c_{ij} = a_i b_j)"_tex` UDL.*

Companion to [`tutorials/01_formula-is-the-program.ipynb`](../../tutorials/01_formula-is-the-program.ipynb) (the C++ xcpp20 tutorial). Same parser, same AST, same `Evaluator` — only the user-facing surface differs: Python has no UDLs, so the access path is `tensor.tex.parse(...)` instead of `R"(...)"_tex`. Phase 6 M5 per [ADR-0018](../../docs/arc42/09-decisions/0018-phase-6-python-sdk-entry-via-nanobind.md).

## Prerequisites

- [`00_python-sdk-tour.ipynb`](./00_python-sdk-tour.ipynb) — `Axis`, `DynamicShape`, `DynamicTensor`, NumPy interop.

## What this notebook covers

1. **`tex.parse`** — turn a string into an `Expression` AST.
2. **`tex.to_latex` round-trip** — `to_latex(parse(s))` reproduces `s` canonically.
3. **`Evaluator`** — bind named tensors to AST leaves, evaluate to a `DynamicTensor`.
4. **Outer product** — `c_{ij} = a_i b_j` end-to-end (the README headline reproduced in Python).
5. **Einstein-sum** — `\sum_i {a_i b_i}` as a one-line inner product.
6. **Guards** — unbound variables / subscript-count mismatches raise.
7. **Where this fits** — milestone map + cross-references.

In [ ]:
# Colab / Binder setup — install the tensor SDK on first run.
try:
    import tensor  # noqa: F401
except ImportError:
    import subprocess as _sp
    _sp.run(
        ["pip", "install", "--quiet",
         "git+https://github.com/uyuutosa/tensor.git"],
        check=True,
    )
    import tensor  # noqa: F401

In [ ]:
import numpy as np

import tensor
import tensor.tex as tex

print(f"tensor.__version__ = {tensor.__version__}")

## 1. `tex.parse` — string → `Expression`

The simplest invocation: parse a string into an AST. The result is structural — no evaluation, no values. Just the parse tree.

The supported LaTeX subset covers:

- **`IndexedVar`** — `a`, `a_i`, `c_{ij}` (subscripts in `{...}` braces or single-character).
- **`BinOp`** — `+`, `-`, `*`, `/`. Juxtaposition (e.g. `a_i b_j`) is implicit multiplication.
- **`Sum`** — `\sum_i {body}` with the index in `_` and the body in `{...}`.
- **`Equation`** — `lhs = rhs`, where the LHS is an `IndexedVar` annotating expected output shape.
- **`Group`** — explicit `{...}` braces, evaluation-transparent.

In [ ]:
expr = tex.parse(r"c_{ij} = a_i + b_j")
print(f"parsed: {expr}")
print(f"empty? {expr.empty()}")
print(f"round-trip via to_latex: {tex.to_latex(expr)}")

## 2. Round-trip property

For every supported expression `e`, `parse(to_latex(e)) == e`. This is what makes the surface **invertible** — you can write LaTeX, the program runs, and you can recover canonical LaTeX from the AST for citation or display.

In [ ]:
samples = [
    r"a_i",
    r"a_i + b_i",
    r"a_i b_j",                  # juxtaposition → implicit multiplication
    r"c_{ij} = a_i + b_j",
    r"\sum_i {a_i b_i}",
]
for src in samples:
    canonical = tex.to_latex(tex.parse(src))
    print(f"  {src:<30s} → {canonical}")

## 3. `Evaluator` — bind + evaluate

Parsing alone is structural. The `Evaluator<T>` (here `tex.Evaluator` for `float64`) closes the loop: bind named tensors to the AST's `IndexedVar` leaves, walk the expression, and produce a concrete `DynamicTensor`. The walk delegates the actual algebra to `tensor::core`'s `DynamicTensor` operators — same Einstein-broadcast machinery the C++ Domain uses.

Below: the canonical outer-product example from the project's README + notebook 01.

In [ ]:
# Two rank-1 tensors with disjoint labels — the broadcast setup.
a = tensor.from_numpy(np.array([1.0, 2.0, 3.0]), ["i"])
b = tensor.from_numpy(np.array([10.0, 20.0]), ["j"])

ev = tex.Evaluator()
ev.bind("a", a)
ev.bind("b", b)

outer = ev.evaluate(tex.parse(r"c_{ij} = a_i b_j"))
print("c = a_i b_j  (outer product):")
print(outer)
print(f"shape rank = {outer.shape.rank()}")

# Cross-validate against NumPy's outer
expected = np.outer([1.0, 2.0, 3.0], [10.0, 20.0])
np.testing.assert_allclose(outer.numpy(), expected, atol=1e-12)

## 4. `\sum_i {a_i b_i}` — one-line Einstein-style inner product

When two tensors share an axis label, `\sum_i {a_i b_i}` is the Einstein-notation inner product. The `Sum` node's body produces a multi-element result (the element-wise product) which `\sum` reduces over the named axis.

In [ ]:
c_np = np.array([1.0, 2.0, 3.0, 4.0])
d_np = np.array([2.0, 3.0, 5.0, 7.0])

ev = tex.Evaluator()
ev.bind("c", tensor.from_numpy(c_np, ["i"]))
ev.bind("d", tensor.from_numpy(d_np, ["i"]))

scalar = ev.evaluate(tex.parse(r"\sum_i {c_i d_i}"))
print(f"\\sum_i c_i d_i = {scalar}")
print(f"  rank-0 scalar value: {scalar[0]}")
print(f"  expected (numpy.dot): {float(np.dot(c_np, d_np))}")

## 5. Equation form — `c_{ij} = a_i + b_j`

An `Equation` node ties an LHS shape annotation (`c_{ij}`) to an RHS expression. The Evaluator computes the RHS and uses the LHS to validate the result's rank + label set.

In [ ]:
a = tensor.from_numpy(np.array([1.0, 2.0, 3.0, 4.0, 5.0]), ["i"])
b = tensor.from_numpy(np.array([10.0, 20.0]), ["j"])

ev = tex.Evaluator()
ev.bind("a", a)
ev.bind("b", b)
result = ev.evaluate(tex.parse(r"c_{ij} = a_i + b_j"))

expected = (
    np.array([1.0, 2.0, 3.0, 4.0, 5.0]).reshape(-1, 1)
    + np.array([10.0, 20.0]).reshape(1, -1)
)
np.testing.assert_allclose(result.numpy(), expected, atol=1e-12)
print("c_{ij} = a_i + b_j (5x2 outer-sum table):")
print(result.numpy())

## 6. Guards — unbound variable / subscript-count mismatch

The Evaluator raises on misuse rather than producing garbage tensors:

In [ ]:
# Unbound variable raises
ev = tex.Evaluator()
try:
    ev.evaluate(tex.parse(r"a_i + b_j"))
except Exception as e:
    print(f"unbound variable correctly raised:\n  {e}")

# Subscript-count mismatch: rank-2 tensor referred to with one subscript
ev = tex.Evaluator()
ev.bind("a", tensor.from_numpy(np.zeros((3, 4)), ["i", "j"]))
try:
    ev.evaluate(tex.parse(r"a_i"))
except Exception as e:
    print(f"\nsubscript-count mismatch correctly raised:\n  {e}")

## 7. Where this fits

**Phase 6 milestone map** (2026-05-13):

| Milestone | Surface | Status |
| --------- | ------- | ------ |
| P6.M1 | scaffold + smoke binding | ✅ |
| P6.M2 | `Axis` + `DynamicShape` + `DynamicTensor` + arithmetic | ✅ |
| P6.M3 | `contract` + NumPy interop | ✅ |
| P6.M4 | autograd (`DynamicVariable`, `backward`, `sgd_update`) | ✅ |
| **P6.M5** | **`tex.parse` + `Evaluator` (this notebook)** | **✅** |
| P6.M6 | runtime backend selection + `0.2.0` release | next |

**Sibling notebooks** (Python):

- [`00_python-sdk-tour.ipynb`](./00_python-sdk-tour.ipynb) — `DynamicTensor` + arithmetic + NumPy interop.
- [`01_python-autograd.ipynb`](./01_python-autograd.ipynb) — autograd + linear-regression training loop.

**Sibling notebook** (C++ xcpp20):

- [`tutorials/01_formula-is-the-program.ipynb`](../../tutorials/01_formula-is-the-program.ipynb) — the C++ side of the same content (`_tex` UDL end-to-end).

**Cross-references**:

- [ADR-0005](../../docs/arc42/09-decisions/0005-adopt-tex-lyx-as-authoring-surface.md) — TeX/LyX as a first-class authoring surface.
- [`docs/detailed-design/tensor-tex.md`](../../docs/detailed-design/tensor-tex.md) — the parser / AST / Evaluator / LyX export implementation HOW.
- [ADR-0018](../../docs/arc42/09-decisions/0018-phase-6-python-sdk-entry-via-nanobind.md) — Phase 6 / Python SDK entry decisions.